# MenuMind: LoRA и сравнение конфигураций

Выполненный учебный эксперимент на синтетических данных. **Human review не выполнен.** Run All проверяет сохранённые ответы без сети и GPU; он не повторяет обучение. Команды обучения и последовательность приведены в README.md. Notebook требует всю папку эксперимента.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
candidates = [Path.cwd(), Path.cwd()/'experiments/lora-study', Path.cwd()/'MenuMind/experiments/lora-study']
ROOT = next(p.resolve() for p in candidates if (p/'config.json').exists() and (p/'audit_artifacts.py').exists())
sys.path.insert(0, str(ROOT))
from audit_artifacts import main as audit
audit()

{'passed': True, 'raw_generations_recomputed': 108, 'split_leakage_checked': True, 'frozen_source_hashes_checked': True, 'selection_uses_validation': True, 'human_review_completed': False}


## Данные и воспроизводимость

Обе языковые версии блюда остаются в одной части. Все эталоны синтетические; общая структура эталона ограничивает перенос выводов на реальные меню.

In [2]:
manifest = json.loads((ROOT/'dataset/manifest.json').read_text(encoding='utf-8'))
print(json.dumps(manifest, ensure_ascii=False, indent=2))

{
  "seed": 42,
  "split_unit": "ingredient-pair dish; both languages stay together",
  "counts": {
    "train": {
      "rows": 168,
      "dishes": 84
    },
    "validation": {
      "rows": 36,
      "dishes": 18
    },
    "test": {
      "rows": 36,
      "dishes": 18
    }
  },
  "sha256": {
    "train": "045be4f66f1d7b2022dc3388af187b2b978dd7419487e1febfde5a3cfa755588",
    "validation": "0548308fd4102cf16efc6a3106071aa771dc5cc6f70be5350e42abcab0474f84",
    "test": "ebb433fb4af06b38fd88c04218a84b11093c00548281b94e50531573175d3c4e"
  },
  "lexicon_sha256": "2d19d0cebfc054b5c126a63423d1c02ae88c72d3113f216d19339a505a5a5bef",
  "source": "New authored synthetic teaching dataset, no API calls, no human annotations",
  "limitations": [
    "Shared vocabulary and target template across splits",
    "Held-out combinations, not unseen domains",
    "No naturalness, native Kazakh, customer, safety or conversion claims",
    "Only one channel and neutral tone"
  ],
  "human_review_comple

## Tuning и настоящее обучение

Выбор learning rate сделан по validation response-token NLL, до генерации на test. Обучаются только LoRA-матрицы.

In [3]:
selection = json.loads((ROOT/'artifacts/selection.json').read_text(encoding='utf-8'))
display(pd.DataFrame([{k:t[k] for k in ['learning_rate','validation_response_token_nll','trainable_parameters','total_parameters','adapter_weight_delta_l2','seconds']} for t in selection['trials']]))
print('Selected LR:', selection['selected']['learning_rate'])
print((ROOT/'artifacts/environment.json').read_text(encoding='utf-8'))

,learning_rate,validation_response_token_nll,trainable_parameters,total_parameters,adapter_weight_delta_l2,seconds
0,0.0002,0.004605,540672,494573440,3.287546,116.950398
1,0.0005,0.001106,540672,494573440,4.664046,200.546597


Selected LR: 0.0005
{
  "python": "3.10.3",
  "torch": "2.6.0+cu124",
  "cuda": "12.4",
  "gpu": "NVIDIA GeForce RTX 4060 Laptop GPU",
  "bf16": true,
  "config": {
    "seed": 42,
    "model_id": "Qwen/Qwen2.5-0.5B-Instruct",
    "revision": "7ae557604adf67be50417f59c2c2f167def9a775",
    "max_sequence_length": 512,
    "max_new_tokens": 128,
    "batch_size": 2,
    "gradient_accumulation_steps": 4,
    "epochs": 3,
    "optimizer": "AdamW",
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "lora_rank": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "target_modules": [
      "q_proj",
      "v_proj"
    ],
    "learning_rates": [
      0.0002,
      0.0005
    ],
    "selection_metric": "validation_response_token_nll",
    "decoding": {
      "do_sample": false,
      "num_beams": 1
    },
    "scope": "Synthetic bilingual single-channel description study; not production deployment or human evaluation"
  },
  "loss": "causal cross-entropy on response tokens only; prompt

## Единая таблица результатов

Разные задачи показаны отдельно. Пустая ячейка означает неприменимость метрики, а не нулевой результат.

In [4]:
results = pd.read_csv(ROOT/'artifacts/results.csv')
display(results)

,task,language,n,model,C,threshold,micro_f1,macro_f1,json_valid,constraint_pass,ingredient_recall,method_match,known_fact_pass,grounded_slot_pass,mean_latency_seconds
0,EN tag classification,en,2651,frequency,NaN,0.20,0.240271,0.031576,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,EN tag classification,en,2651,classical_default,1.0,0.50,0.487600,0.376710,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,EN tag classification,en,2651,classical_tuned,2.0,0.35,0.590696,0.585985,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,RU/KK synthetic description,all,36,base_simple,NaN,NaN,NaN,NaN,0.444444,0.416667,0.222222,0.138889,0.416667,0.055556,5.884050
4,RU/KK synthetic description,ru,18,base_simple,NaN,NaN,NaN,NaN,0.888889,0.833333,0.444444,0.277778,0.833333,0.111111,3.422898
5,RU/KK synthetic description,kk,18,base_simple,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.345201
6,RU/KK synthetic description,all,36,base_detailed,NaN,NaN,NaN,NaN,0.527778,0.416667,0.458333,0.277778,0.527778,0.138889,6.123631
7,RU/KK synthetic description,ru,18,base_detailed,NaN,NaN,NaN,NaN,0.944444,0.722222,0.833333,0.444444,0.944444,0.222222,4.520059
8,RU/KK synthetic description,kk,18,base_detailed,NaN,NaN,NaN,NaN,0.111111,0.111111,0.083333,0.111111,0.111111,0.055556,7.727204
9,RU/KK synthetic description,all,36,lora_detailed,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,4.249949


## Неопределённость и интерпретация

Bootstrap по блюдам сохраняет связанность RU/KK. Это не доверительный интервал по разным training seeds.

In [5]:
llm = json.loads((ROOT/'artifacts/llm_results.json').read_text(encoding='utf-8'))
display(pd.DataFrame(llm['comparisons']))
print((ROOT/'REPORT.txt').read_text(encoding='utf-8'))

,comparison,metric,delta,ci95,resampling_unit,replicates
0,base_detailed minus base_simple,constraint_pass,0.000000,"[-0.1388888888888889, 0.13958333333332956]",dish (RU/KK paired),2000
1,base_detailed minus base_simple,ingredient_recall,0.236111,"[0.1388888888888889, 0.3333333333333333]",dish (RU/KK paired),2000
2,base_detailed minus base_simple,grounded_slot_pass,0.083333,"[-0.027777777777777776, 0.19444444444444445]",dish (RU/KK paired),2000
3,lora_detailed minus base_detailed,constraint_pass,0.583333,"[0.4722222222222222, 0.6944444444444444]",dish (RU/KK paired),2000
4,lora_detailed minus base_detailed,ingredient_recall,0.541667,"[0.5, 0.5833333333333334]",dish (RU/KK paired),2000
5,lora_detailed minus base_detailed,grounded_slot_pass,0.861111,"[0.75, 0.9444444444444444]",dish (RU/KK paired),2000


MenuMind: выполненный учебный эксперимент с LoRA

Область выводов: синтетические RU/KK-описания одного канала, нейтральный тон. Это не проверка коммерческого качества или казахской грамматики человеком.

Данные: 120 синтетических блюд, 240 языковых примеров. Train 84 блюда / 168 строк; validation 18 / 36; test 18 / 36. Разделение по паре основных ингредиентов, seed=42. Переводы одного блюда не пересекают split. Словарь и шаблон эталона общие — эксперимент проверяет новые комбинации знакомых элементов.

Классический ML: те же опубликованные train/validation/test, что в tags-midterm. Новый baseline C=1, threshold=0.5. Поиск C=[0.5,2], threshold=[0.2,0.35,0.5]; выбор по validation micro-F1. Старый test уже был просмотрен, поэтому это дополнение к прежнему сравнению, а не новый закрытый holdout.
До tuning: micro-F1=0.4876, macro-F1=0.3767.
После tuning: micro-F1=0.5907, macro-F1=0.5860; C=2.0, threshold=0.35.
Прирост micro-F1=0.1031; macro-F1=0.2093. Менялись регуляризация и порог; без отд

## Примеры для проверки человеком

Поля human_review.csv оставлены пустыми. Автоматическая словарная проверка не является оценкой естественности, грамматики или всех неподтверждённых утверждений.

In [6]:
review = pd.read_csv(ROOT/'artifacts/human_review.csv')
display(review[['id','model','language','recipe','output']].head(9))
assert review[['facts_correct','language_correct','naturalness_1_to_5','reviewer']].isna().all().all()

,id,model,language,recipe,output
0,dish-00-03-ru,base_simple,ru,"{""name"": ""Курица и кабачок"", ""ingredients"": [""...","{\n ""description"": ""Вот рецепт простого блюда..."
1,dish-00-03-kk,base_simple,kk,"{""name"": ""Тауық еті және кәді"", ""ingredients"":...","{\n ""description"": ""Тауық еті және кәді - тау..."
2,dish-00-04-ru,base_simple,ru,"{""name"": ""Курица и тыква"", ""ingredients"": [""ку...","{\n ""description"": ""Сладкий курицей и тыквой,..."
3,dish-00-04-kk,base_simple,kk,"{""name"": ""Тауық еті және асқабақ"", ""ingredient...","{\n ""description"": ""Күнүн тауық еті және асқа..."
4,dish-01-01-ru,base_simple,ru,"{""name"": ""Говядина и гречка"", ""ingredients"": [...","{\n ""description"": ""Сладкий говядине с гречко..."
5,dish-01-01-kk,base_simple,kk,"{""name"": ""Сиыр еті және қарақұмық"", ""ingredien...","{\n ""description"": ""Сиыр еті және қарақұмық -..."
6,dish-01-03-ru,base_simple,ru,"{""name"": ""Говядина и кабачок"", ""ingredients"": ...","{\n ""description"": ""Сладкий и вкусный говядин..."
7,dish-01-03-kk,base_simple,kk,"{""name"": ""Сиыр еті және кәді"", ""ingredients"": ...","{\n ""description"": ""Күн жана көп, күн және кө..."
8,dish-01-04-ru,base_simple,ru,"{""name"": ""Говядина и тыква"", ""ingredients"": [""...","{\n ""description"": ""Сладкий и вкусный говядин..."
